# Línea A: exploración léxica y de entidades en las fuentes primarias

**Corpus:** los 7 volúmenes de *La Rebelión de Túpac Amaru II* (Colección Documental, Congreso del Perú), **solo fuentes primarias**: se excluyen el frontispicio editorial (presentaciones, prólogos de Valcárcel y Walker, presentación de cada volumen por Huerto), el colofón y las notas al pie.
**Método:** recuento de términos, expresiones de fecha y listas de nombres (gazetteers) sobre el texto OCR.
No hay modelos de lenguaje ni entrenamiento: cada cifra se puede rastrear a las páginas de origen con `kwic()`.

| Pregunta | Sección |
|---|---|
| ¿Qué periodo cubre cada volumen? | 2 |
| ¿Con qué palabras se designa al movimiento y a sus protagonistas? | 3 |
| ¿Cuándo se produce la documentación? | 4 |
| ¿Qué lugares y qué personas aparecen, y con quién se relacionan? | 5 y 6 |

## Límites que condicionan cualquier conclusión
1. **OCR con errores.** Las cifras son un *piso*: las variantes mal reconocidas no se cuentan.
2. **Una edición, no el archivo.** Los volúmenes son una selección hecha por editores (Valcárcel en los años 70;
   reedición del Bicentenario). Que algo no aparezca no significa que no existiera.
3. **Voz de época.** Casi todo es lenguaje de las autoridades coloniales y de los procesos judiciales:
   los términos dicen **cómo nombraba el poder**, no cómo se nombraba el movimiento a sí mismo.
4. **Texto moderno residual.** El frontispicio, el colofón y las notas al pie se excluyen con las columnas `rol` y
   `texto_fuente` del corpus. Puede quedar alguna nota editorial que continúe en la página siguiente (sin separador);
   usa `kwic()` para distinguirla de la voz de época.
5. **Gazetteers incompletos.** Las listas de lugares y personas son una propuesta: edítalas y vuelve a ejecutar.

In [ ]:
from pathlib import Path
import re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

DATA = Path("data")
OUT = DATA / "linea_a"
OUT.mkdir(parents=True, exist_ok=True)

# Estilo sobrio: sin marco superior/derecho, rejilla tenue, una sola tinta de texto.
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
    "font.size": 10, "axes.titleweight": "bold", "axes.titlelocation": "left",
})
AZUL, NARANJA, GRIS = "#2a6fb0", "#d97a1f", "#6b7280"   # azul = magnitud / personas, naranja = lugares

## 1. Cargar y normalizar el texto

In [ ]:
def norm(t: str) -> str:
    """Minúsculas, sin tildes y uniendo palabras cortadas por guion de fin de línea ("Doc- trina")."""
    t = re.sub(r"(\w)- (\w)", r"\1\2", str(t))
    t = unicodedata.normalize("NFKD", t.lower())
    return "".join(c for c in t if not unicodedata.combining(c))

corpus = pd.read_csv(DATA / "tupac_amaru_corpus.csv", encoding="utf-8-sig")
src = corpus[(corpus.rol == "fuente_primaria") & corpus.documento.str.contains(r"vol\. \d")].copy()
src["vol"] = src.documento.str.extract(r"vol\. (\d)")[0].astype(int)
src["t_bruto"] = src.texto_fuente.map(norm)   # cuerpo de la página, sin notas al pie
VOLS = sorted(src.vol.unique())

### 1.1 Encabezados corridos (ruido que hay que quitar)

Las páginas repiten dos encabezados que **no son contenido**: el de la colección
(*«Nueva Colección Documental de la Independencia del Perú — La Rebelión de Túpac Amaru II»*) y un título de sección
(*«Volumen 6 · Causas criminales seguidas contra los rebeldes»*). Sin quitarlos, "rebelión", "rebelde" y "Túpac Amaru"
aparecen inflados en cientos de páginas. El título de sección se conserva como metadato (`seccion`): sirve para saber
en qué parte del volumen estás.

In [ ]:
from collections import Counter

ENCABEZADO_COLECCION = re.compile(r"nueva coleccion documental de la independencia del peru\s*(?:la rebelion de tupac amaru ii)?")

def separar_titulo_seccion(textos: pd.Series, minimo: int = 6) -> tuple[pd.Series, pd.Series]:
    """Detecta el título corrido «volumen N <título>»: el prefijo más largo que se repite en >= `minimo` páginas."""
    palabras = textos.map(str.split)
    cuenta: Counter = Counter()
    for w in palabras:
        if len(w) > 3 and w[0] == "volumen" and w[1].isdigit():
            for k in range(3, min(len(w), 17)):
                cuenta[tuple(w[2:k])] += 1
    titulos, limpios = [], []
    for w in palabras:
        titulo = ()
        if len(w) > 3 and w[0] == "volumen" and w[1].isdigit():
            # Se alarga el título palabra por palabra mientras la extensión siga siendo frecuente
            # (>= 15 % de las páginas que comparten el prefijo): así no se arrastra texto real ("…rebelión de").
            for k in range(3, min(len(w), 17)):
                actual, previo = cuenta[tuple(w[2:k])], cuenta[tuple(w[2:k - 1])] if k > 3 else None
                if actual < minimo or (previo is not None and actual < 0.15 * previo):
                    break
                titulo = tuple(w[2:k])
        titulos.append(" ".join(titulo))
        limpios.append(" ".join(w[2 + len(titulo):]) if titulo else " ".join(w))
    return pd.Series(titulos, index=textos.index), pd.Series(limpios, index=textos.index)

sin_a = src.t_bruto.str.replace(ENCABEZADO_COLECCION, " ", regex=True)
src["seccion"], src["t"] = None, None
for v in VOLS:                                   # el título corrido es propio de cada volumen
    idx = src.vol == v
    src.loc[idx, "seccion"], src.loc[idx, "t"] = separar_titulo_seccion(sin_a[idx])

tam = src.groupby("vol").agg(paginas=("pagina", "size"),
                             caracteres_brutos=("t_bruto", lambda s: s.str.len().sum()),
                             caracteres_limpios=("t", lambda s: s.str.len().sum()))
tam["quitado_%"] = ((1 - tam.caracteres_limpios / tam.caracteres_brutos) * 100).round(1)
tam

In [ ]:
# Títulos de sección detectados (verifica que tengan sentido): veces que aparece cada uno
(src[src.seccion != ""].groupby(["vol", "seccion"]).size().rename("paginas")
    .reset_index().sort_values(["vol", "paginas"], ascending=[True, False]).groupby("vol").head(6))

### Concordancias (`kwic`)
Herramienta para **verificar**: muestra cada aparición de un patrón con su contexto, volumen y página.
El texto está normalizado (sin tildes, minúsculas); la página permite ir al original.

In [ ]:
def kwic(patron: str, vol: int | None = None, n: int = 8, ancho: int = 110, seed: int = 0) -> pd.DataFrame:
    """Devuelve n concordancias (aleatorias, reproducibles) del patrón regex, opcionalmente de un volumen."""
    rx = re.compile(patron)
    base = src if vol is None else src[src.vol == vol]
    filas = []
    for r in base.itertuples():
        for m in rx.finditer(r.t):
            a, b = max(0, m.start() - ancho), min(len(r.t), m.end() + ancho)
            filas.append({"vol": r.vol, "pagina": r.pagina, "contexto": "…" + r.t[a:b] + "…"})
    df = pd.DataFrame(filas)
    print(f"{len(df)} apariciones de /{patron}/" + (f" en vol. {vol}" if vol else ""))
    return df.sample(min(n, len(df)), random_state=seed).sort_values(["vol", "pagina"]) if len(df) else df

def por_volumen(patrones: dict[str, str], por: float = 1e5) -> pd.DataFrame:
    """Frecuencia de cada familia de términos por volumen, por cada `por` caracteres (comparable entre volúmenes)."""
    filas = {}
    for v in VOLS:
        txt = " ".join(src[src.vol == v].t)
        filas[v] = {k: len(re.findall(p, txt)) / len(txt) * por for k, p in patrones.items()}
    return pd.DataFrame(filas).rename_axis("término").rename(columns=lambda v: f"vol. {v}")

def mapa_calor(df: pd.DataFrame, titulo: str, fmt: str = "{:.0f}", archivo: str | None = None):
    """Mapa de calor de un solo tono (magnitud): más oscuro = más frecuente. Los valores van impresos."""
    fig, ax = plt.subplots(figsize=(1.0 * df.shape[1] + 3.5, 0.38 * df.shape[0] + 1.4))
    im = ax.imshow(df.values, cmap="Blues", aspect="auto")
    ax.set_xticks(range(df.shape[1]), df.columns); ax.set_yticks(range(df.shape[0]), df.index)
    ax.grid(False); ax.xaxis.tick_top()
    umbral = np.nanmax(df.values) * 0.55
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            val = df.values[i, j]
            ax.text(j, i, fmt.format(val), ha="center", va="center", fontsize=9,
                    color="white" if val > umbral else "#1f2937")
    ax.set_title(titulo, pad=28)
    fig.tight_layout()
    if archivo:
        fig.savefig(OUT / archivo, bbox_inches="tight")
    plt.show()

## 2. ¿Qué periodo cubre cada volumen?

Antes de comparar volúmenes hay que saber **de qué momento hablan**. Se cuentan los años (1775–1795) que
aparecen escritos en cifras. Los volúmenes mezclan etapas distintas del proceso, y eso condiciona toda comparación.

In [ ]:
ANIOS = list(range(1775, 1796))
filas = {}
for v in VOLS:
    txt = " ".join(src[src.vol == v].t)
    c = pd.Series(re.findall(r"\b(17[7-9]\d)\b", txt)).value_counts()
    filas[f"vol. {v}"] = {a: int(c.get(str(a), 0)) for a in ANIOS}
anios = pd.DataFrame(filas).rename_axis("año")
anios.to_csv(OUT / "anios_por_volumen.csv", encoding="utf-8-sig")
mapa_calor(anios.T, "Años mencionados por volumen (n.º de menciones)", fmt="{:.0f}", archivo="01_anios_por_volumen.png")

> **Lectura:** el vol. 5 casi no trae años en cifras (las fechas van en letras), así que queda subrepresentado en las
> secciones de cronología. Los volúmenes 6 y 7 se concentran en 1781 (procesos y sentencias).

## 3. Vocabulario con que se designa al movimiento y a sus protagonistas

Frecuencia por cada 100 000 caracteres (para que los volúmenes, de distinto tamaño, sean comparables).
Cada familia de términos es un patrón regex: puedes editarlas.

In [ ]:
DESIGNACIONES = {
    # cómo se nombra el hecho
    "rebelión":              r"\brebeli",
    "levantamiento":         r"\blevantamient",
    "sublevación / sublevado": r"\bsublev",
    "alzamiento / alzado":   r"\balzamient|\balzad",
    "sedición / tumulto":    r"\bsedicion|\btumult|\bconmocion",
    "insurrección / insurgente": r"\binsurrec|\binsurgen",
    "revolución":            r"\brevoluci",
    # cómo se nombra a los protagonistas
    "rebelde":               r"\brebeld",
    "traidor / traición":    r"\btraidor|\btraicion",
    "tirano":                r"\btirano",
    "caudillo":              r"\bcaudillo",
    "bandido / facineroso":  r"\bbandid|\bfacinero|\bmalhechor",
    "inca":                  r"\b[iy]nca\b",
    # marcos de legitimidad
    "rey / majestad":        r"\brey\b|\bmajestad",
    "libertad":              r"\blibertad",
    "justicia":              r"\bjusticia",
}
desig = por_volumen(DESIGNACIONES)
desig.round(1).to_csv(OUT / "designaciones_por_volumen.csv", encoding="utf-8-sig")
mapa_calor(desig, "Designaciones y marcos de legitimidad por 100 000 caracteres", fmt="{:.0f}",
           archivo="02_designaciones.png")

### 3.1 "Rebelión" frente a "revolución"
Es el eje del debate historiográfico. En las fuentes de época la balanza es muy desigual; la pregunta es
**quién** usa "revolución" y en qué sentido. Las concordancias lo muestran.

In [ ]:
pd.set_option("display.max_colwidth", 260); pd.set_option("display.width", 250)
conteo = pd.DataFrame({
    "rebelión (apariciones)":  {v: len(re.findall(r"\brebeli", " ".join(src[src.vol == v].t))) for v in VOLS},
    "revolución (apariciones)": {v: len(re.findall(r"\brevoluci", " ".join(src[src.vol == v].t))) for v in VOLS},
})
conteo.index = [f"vol. {v}" for v in conteo.index]
conteo

In [ ]:
# ¿Quién dice "revolución"? Leer en contexto (distinguir voz de época de prólogo/nota de los editores).
kwic(r"\brevoluci", n=15)

In [ ]:
# ...y los usos de "rebelión", en una muestra, para comparar el sentido
kwic(r"\brebeli", n=10)

### 3.2 ¿Cómo se nombra a Túpac Amaru?
Estas concordancias buscan las fórmulas con que las autoridades se refieren a él (p. ej. "el rebelde José Gabriel").

In [ ]:
kwic(r"(?:rebelde|traidor|tirano|caudillo|falso inca|mal llamado|llamado)\s+(?:jose gabriel|tupac|inca|condorcanqui)", n=12)

## 4. Cronología de la documentación

Se extraen las fechas completas (`día de mes de 17xx`). Es la fecha **mencionada en el texto**; no siempre es la fecha
del documento (un oficio puede referirse a hechos anteriores). Úsala como indicio de la actividad documental.

In [ ]:
MES_NUM = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7, "agosto": 8,
           "setiembre": 9, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}
rx_fecha = re.compile(rf"\b(\d{{1,2}})\s+de\s+({'|'.join(MES_NUM)})\s+de\s+(17\d\d)\b")
filas = [
    {"vol": r.vol, "pagina": r.pagina, "anio": int(anio), "mes": MES_NUM[mes], "dia": int(dia)}
    for r in src.itertuples() for dia, mes, anio in rx_fecha.findall(r.t)
]
fechas = pd.DataFrame(filas)
fechas["periodo"] = pd.to_datetime({"year": fechas.anio, "month": fechas.mes, "day": 1}).dt.to_period("M")
print(len(fechas), "fechas completas encontradas")
fechas.groupby("vol").size().to_frame("fechas")

In [ ]:
ventana = fechas[(fechas.anio >= 1780) & (fechas.anio <= 1784)]
por_mes = ventana.groupby("periodo").size()
idx = pd.period_range("1780-01", "1784-12", freq="M")
por_mes = por_mes.reindex(idx, fill_value=0)
por_mes.rename("fechas").to_csv(OUT / "fechas_por_mes.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(range(len(por_mes)), por_mes.values, color=AZUL, width=0.8)
marcas = [i for i, p in enumerate(por_mes.index) if p.month == 1]
ax.set_xticks(marcas, [str(por_mes.index[i].year) for i in marcas])
ax.set_ylabel("fechas mencionadas"); ax.set_title("Fechas mencionadas por mes, 1780–1784")
pico = por_mes.idxmax()
ax.annotate(f"máximo: {pico} ({por_mes.max()})", xy=(list(por_mes.index).index(pico), por_mes.max()),
            xytext=(8, -2), textcoords="offset points", color="#1f2937")
fig.tight_layout(); fig.savefig(OUT / "03_cronologia.png", bbox_inches="tight"); plt.show()

In [ ]:
# Volumen x mes (1780-1782): permite ver qué volumen documenta qué momento
tabla = (ventana[ventana.anio <= 1782].pivot_table(index="periodo", columns="vol", values="dia", aggfunc="size", fill_value=0))
tabla.columns = [f"vol. {c}" for c in tabla.columns]
tabla

## 5. Lugares

Lista de topónimos de la región (editable). Se cuentan **menciones**; "canas" y "lares" pueden confundirse con palabras comunes;
revísalas con `kwic()` antes de interpretarlas ("tinta" se comprobó: casi siempre es el pueblo).

In [ ]:
LUGARES = {
    "Cusco":            r"\bcuzco\b|\bcusco\b",
    "Tinta":            r"\bpueblo de tinta\b|\bprovincia de tinta\b|\btinta\b",
    "Tungasuca":        r"\btungasuca\b",
    "Pampamarca":       r"\bpampamarca\b",
    "Sangarará":        r"\bsangarara\b",
    "Quispicanchis":    r"\bquispicanchi",
    "Canas y Canchis":  r"\bcanas\b|\bcanchis\b",
    "Chumbivilcas":     r"\bchumbivilca",
    "Paucartambo":      r"\bpaucartambo\b",
    "Calca / Lares":    r"\bcalca\b|\blares\b",
    "Urubamba / Yucay": r"\burubamba\b|\byucay\b",
    "Abancay / Apurímac": r"\babancay\b|\bapurimac\b",
    "Lampa":            r"\blampa\b",
    "Azángaro":         r"\bazangaro\b",
    "Carabaya":         r"\bcarabaya\b",
    "Puno":             r"\bpuno\b",
    "Chucuito":         r"\bchucuito\b",
    "La Paz":           r"\bla paz\b",
    "Oruro":            r"\boruro\b",
    "Potosí":           r"\bpotosi\b",
    "Arequipa":         r"\barequipa\b",
    "Huamanga":         r"\bguamanga\b|\bhuamanga\b",
    "Huancavelica":     r"\bhuancavelica\b",
    "Lima":             r"\blima\b",
    "Buenos Aires":     r"\bbuenos aires\b",
}

def conteo_entidades(dic: dict[str, str]) -> pd.DataFrame:
    filas = {}
    for v in VOLS:
        txt = " ".join(src[src.vol == v].t)
        filas[f"vol. {v}"] = {k: len(re.findall(p, txt)) for k, p in dic.items()}
    df = pd.DataFrame(filas)
    df["total"] = df.sum(axis=1)
    return df.sort_values("total", ascending=False)

def barras(serie: pd.Series, titulo: str, color: str, archivo: str, n: int = 15):
    s = serie.head(n)[::-1]
    fig, ax = plt.subplots(figsize=(7.5, 0.34 * len(s) + 1.2))
    ax.barh(s.index, s.values, color=color, height=0.65)
    for y, v in enumerate(s.values):
        ax.text(v, y, f" {v:,}", va="center", fontsize=9, color="#1f2937")
    ax.grid(axis="y", visible=False); ax.set_title(titulo)
    ax.set_xlim(0, s.max() * 1.12)
    fig.tight_layout(); fig.savefig(OUT / archivo, bbox_inches="tight"); plt.show()

lugares = conteo_entidades(LUGARES)
lugares.to_csv(OUT / "lugares_por_volumen.csv", encoding="utf-8-sig")
barras(lugares["total"], "Lugares más mencionados (todos los volúmenes)", NARANJA, "04_lugares.png")
lugares

## 6. Personas

Lista de actores principales, con sus variantes ortográficas habituales (editable). Incluye
**bando rebelde**, **autoridades realistas** y **otras rebeliones andinas** contemporáneas.

> **Ojo con "Túpac Amaru / José Gabriel":** el patrón cuenta la cadena *Túpac Amaru*, que también llevan sus parientes (Diego Cristóbal, Andrés, Fernando, Hipólito...) y que aparece en frases como "el movimiento de Túpac Amaru". No equivale a menciones de la persona de José Gabriel; para eso hay que leer con `kwic()`.

In [ ]:
PERSONAS = {
    # bando rebelde
    "Túpac Amaru / José Gabriel": r"\bt[h]?upa+[ck]?[\s\-]*amar[uo]|\bjose[fph]?\s+gabriel\b|\bcondorcanqui\b",
    "Micaela Bastidas":           r"\bmica[ei]la\b|\bbastid[ae]s\b",
    "Diego Cristóbal":            r"\bdiego\s+(?:chr?isto[bv]al|xpto[bv]al)",
    "Andrés Mendigure":           r"\bmendigu?r[ei]\b|\bmendiguir[ei]\b",
    "Hipólito Túpac Amaru":       r"\bhipolito\b",
    "Tomasa Tito Condemayta":     r"\btomasa\b|\bcondema[iy]ta\b",
    "Francisco Túpac Amaru":      r"\bfrancisco tupac\b|\bfernando tupac\b",
    # autoridades coloniales
    "José Antonio de Areche":     r"\bareche\b",
    "Benito de la Mata Linares":  r"\bmata linares\b",
    "José del Valle":             r"\bjose del valle\b|\bdel valle\b",
    "Gabriel de Avilés":          r"\baviles\b",
    "Ventura Landaeta":           r"\blandaeta\b",
    "Antonio de Arriaga":         r"\barriaga\b",
    "Agustín de Jáuregui":        r"\bjauregui\b",
    "Juan Manuel de Moscoso":     r"\bmoscoso\b",
    "Visitador Escobedo":         r"\bescobedo\b",
    # otras figuras / rebeliones
    "Túpac Katari / Julián Apaza": r"\bcatari\b|\bkatari\b|\bjulian apaza\b",
    "Pedro Vilca Apaza":          r"\bvilca apaza\b|\bvilcapaza\b",
    "Mateo Pumacahua":            r"\bpumacahua\b",
}
personas = conteo_entidades(PERSONAS)
personas.to_csv(OUT / "personas_por_volumen.csv", encoding="utf-8-sig")
barras(personas["total"], "Personas más mencionadas (todos los volúmenes)", AZUL, "05_personas.png", n=19)
personas

## 7. Red de coocurrencias (personas y lugares)

Dos entidades están conectadas si **aparecen en la misma página**; el peso es el número de páginas compartidas.
Es una señal de proximidad en el texto, **no** de una relación histórica demostrada. Túpac Amaru aparece en la mayoría de las páginas, así que se conecta con casi todo: lo informativo son los vínculos **entre los demás** nodos.

In [ ]:
ENTIDADES = {**{f"P:{k}": v for k, v in PERSONAS.items()}, **{f"L:{k}": v for k, v in LUGARES.items()}}
rx_ent = {k: re.compile(p) for k, p in ENTIDADES.items()}

pares: dict[tuple[str, str], int] = {}
menciones: dict[str, int] = {}
for t in src.t:
    presentes = sorted(k for k, rx in rx_ent.items() if rx.search(t))
    for k in presentes:
        menciones[k] = menciones.get(k, 0) + 1
    for i, a in enumerate(presentes):
        for b in presentes[i + 1:]:
            pares[(a, b)] = pares.get((a, b), 0) + 1

aristas = (pd.DataFrame([{"origen": a, "destino": b, "paginas": w} for (a, b), w in pares.items()])
             .sort_values("paginas", ascending=False).reset_index(drop=True))
aristas.to_csv(OUT / "coocurrencias.csv", index=False, encoding="utf-8-sig")
print(len(aristas), "pares; páginas con al menos una entidad:", sum(any(rx.search(t) for rx in rx_ent.values()) for t in src.t))
aristas.head(15)

In [ ]:
import matplotlib.patheffects as pe

UMBRAL = 30                                       # páginas compartidas mínimas para dibujar un vínculo
# Nodos "concentradores": aparecen en casi todas las páginas y colapsan el dibujo en una bola. Se omiten
# del dibujo (siguen en coocurrencias.csv); pon [] para incluirlos.
CONCENTRADORES = ["P:Túpac Amaru / José Gabriel", "L:Cusco"]
G = nx.Graph()
for (a, b), w in pares.items():
    if w >= UMBRAL and a not in CONCENTRADORES and b not in CONCENTRADORES:
        G.add_edge(a, b, weight=w)
G.remove_nodes_from([n for n in list(G) if G.degree(n) == 0])

pos = nx.spring_layout(G, weight="weight", k=2.2 / np.sqrt(len(G)), seed=11, iterations=400)
fig, ax = plt.subplots(figsize=(12, 8.5))
maxw = max(d["weight"] for _, _, d in G.edges(data=True))
nx.draw_networkx_edges(G, pos, ax=ax, edge_color=GRIS, alpha=0.30,
                       width=[0.4 + 5.5 * G[u][v]["weight"] / maxw for u, v in G.edges])
nx.draw_networkx_nodes(G, pos, ax=ax, edgecolors="white", linewidths=1.5,
                       node_color=[AZUL if n.startswith("P:") else NARANJA for n in G],
                       node_size=[40 + 22 * np.sqrt(menciones[n]) for n in G])
halo = [pe.withStroke(linewidth=3, foreground="white")]
for n, (x, y) in pos.items():
    ax.annotate(n[2:], (x, y), xytext=(0, 9), textcoords="offset points", ha="center",
                fontsize=8.5, color="#1f2937", path_effects=halo)
ax.set_axis_off(); ax.grid(False)
ax.set_title(f"Coocurrencia en la misma página: personas (azul) y lugares (naranja) · vínculos con ≥ {UMBRAL} páginas",
             loc="left")
ax.text(0, 0, "tamaño del nodo = páginas donde aparece · grosor = páginas compartidas · "
        f"{G.number_of_nodes()} nodos, {G.number_of_edges()} vínculos · no se dibujan: "
        + ", ".join(n[2:] for n in CONCENTRADORES), transform=ax.transAxes, fontsize=8, color=GRIS)
fig.tight_layout(); fig.savefig(OUT / "06_red.png", bbox_inches="tight"); plt.show()

## 8. Cómo usar estos resultados

1. **Verifica antes de citar.** Cualquier cifra debe contrastarse con `kwic()` y con la página original.
2. **Compara con cuidado.** Un volumen con más "rebelde" puede reflejar que contiene más sumarios judiciales, no un cambio de opinión.
3. **Edita los gazetteers.** Añade variantes ortográficas que veas en las concordancias y vuelve a ejecutar.
4. **Registra el método** (versiones de corpus y patrones) si lo vas a presentar: los CSV de `data/linea_a/` son reproducibles.

Siguiente paso posible: añadir coordenadas a los lugares para un mapa de difusión, o separar la voz editorial
(prólogos y notas) de los documentos de época.